# BCI-2: GPU Training on Google Colab with W&B Tracking

This notebook trains all 4 decoder models (GRU, CNN-LSTM, Transformer, CNN-Transformer)
on the Willett handwriting BCI dataset using Colab's free GPU, with full Weights & Biases logging.

**What you need before running:**
1. Your BCI-2 project uploaded to Google Drive (src/, scripts/, requirements.txt, bci_training_data.zip)
2. A free [wandb.ai](https://wandb.ai) account
3. This notebook opened in Google Colab with GPU runtime selected

## 1. Setup: Mount Drive & Install Dependencies

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# CONFIGURE THESE PATHS
# ============================================================
# Path to your BCI-2 project folder in Google Drive.
# After uploading, it should contain:
#   My Drive/BCI-2/src/
#   My Drive/BCI-2/scripts/
#   My Drive/BCI-2/requirements.txt
#   My Drive/BCI-2/bci_training_data.zip
DRIVE_PROJECT_ROOT = '/content/drive/MyDrive/BCI-2'

# W&B settings
WANDB_PROJECT = 'brain-text-decoder'
WANDB_ENTITY = None  # Set to your W&B username or team name, or leave None

In [ ]:
# Copy project from Drive to local SSD for faster I/O during training
import shutil
import os
import zipfile

LOCAL_PROJECT = '/content/BCI-2'

if os.path.exists(LOCAL_PROJECT):
    shutil.rmtree(LOCAL_PROJECT)

print('Copying project from Drive to local storage...')
shutil.copytree(DRIVE_PROJECT_ROOT, LOCAL_PROJECT)

# Unzip the training data into the right location
zip_path = os.path.join(LOCAL_PROJECT, 'bci_training_data.zip')
data_dir = os.path.join(LOCAL_PROJECT, 'data')

if os.path.exists(zip_path):
    print(f'Unzipping training data ({os.path.getsize(zip_path) / 1e6:.0f} MB)...')
    os.makedirs(data_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(data_dir)
    os.remove(zip_path)  # Free up space
    print('Data unzipped!')
else:
    print('WARNING: bci_training_data.zip not found! Make sure you uploaded it to Drive.')

# CRITICAL: Remove stale trial_index.csv (has Windows paths from local machine)
# and rebuild it with correct Colab paths using build_trial_index_from_dir()
stale_csv = os.path.join(data_dir, 'willett_handwriting', 'trial_index.csv')
if os.path.exists(stale_csv):
    os.remove(stale_csv)
    print('Removed stale trial_index.csv (will rebuild with Colab paths)')

os.chdir(LOCAL_PROJECT)
print(f'\nProject at {LOCAL_PROJECT}:')
!ls
print(f'\nData directory:')
!ls data/willett_handwriting/

In [ ]:
# Install dependencies (most are pre-installed on Colab)
!pip install -q wandb jiwer mne transformers pyyaml

In [ ]:
# Verify GPU is available
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected! Go to Runtime > Change runtime type > GPU')

## 2. W&B Login

In [ ]:
import wandb
wandb.login()

## 3. Verify Data & Project Setup

In [ ]:
import sys
sys.path.insert(0, LOCAL_PROJECT)

from src.data.loader import build_trial_index_from_dir

# Rebuild trial index with correct Colab paths (not stale Windows paths)
dataset_dir = os.path.join(LOCAL_PROJECT, 'data', 'willett_handwriting')
print('Building trial index from .npy files...')
trial_index = build_trial_index_from_dir(dataset_dir)

# Cache it for the training script to pick up
index_path = os.path.join(dataset_dir, 'trial_index.csv')
trial_index.to_csv(index_path, index=False)
print(f'Saved trial index: {len(trial_index)} trials')
print(f'Columns: {list(trial_index.columns)}')

if 'n_timesteps' in trial_index.columns:
    ts = trial_index['n_timesteps']
    print(f'\nSignal lengths:')
    print(f'  min={ts.min()}, max={ts.max()}, mean={ts.mean():.0f}, median={ts.median():.0f}')
    print(f'  Trials > 2000: {(ts > 2000).sum()}')
    print(f'  Trials > 5000: {(ts > 5000).sum()}')

## 4. Training Configuration

We train all 4 models with GPU-appropriate settings:
- **Full-size models** (not the reduced CPU versions)
- **t_max=5000** to capture most sentence signals without excessive truncation
- **Mixed precision (fp16)** for faster GPU training
- **All trials** (letters + sentences) with length filtering

In [ ]:
# Training configurations for each model
# Adjust batch sizes based on your GPU memory (T4=16GB, V100=16GB, A100=40GB)

TRAINING_CONFIGS = {
    'gru_decoder': {
        'epochs': 200,
        'batch_size': 16,
        'lr': 3e-4,
        't_max': 5000,
        'hidden_size': 512,
        'n_layers': 3,
        'dropout': 0.3,
        'tags': ['baseline', 'gru', 'gpu', 'full-size'],
    },
    'cnn_lstm': {
        'epochs': 200,
        'batch_size': 12,
        'lr': 3e-4,
        't_max': 5000,
        'hidden_size': 512,
        'n_layers': 2,
        'dropout': 0.5,
        'tags': ['cnn-lstm', 'gpu', 'full-size'],
    },
    'transformer': {
        'epochs': 200,
        'batch_size': 8,  # Transformers use more memory
        'lr': 1e-4,       # Lower LR for transformers
        't_max': 4096,    # Match max_seq_len
        'n_layers': 6,
        'dropout': 0.1,
        'tags': ['transformer', 'gpu', 'full-size'],
    },
    'cnn_transformer': {
        'epochs': 200,
        'batch_size': 8,
        'lr': 1e-4,
        't_max': 4096,
        'n_layers': 4,
        'dropout': 0.1,
        'tags': ['hybrid', 'cnn-transformer', 'gpu', 'full-size'],
    },
}

# Models to train (comment out any you want to skip)
MODELS_TO_TRAIN = [
    'gru_decoder',
    'cnn_lstm',
    'transformer',
    'cnn_transformer',
]

## 5. Train All Models

In [ ]:
import subprocess
import time

results = {}

for model_name in MODELS_TO_TRAIN:
    config = TRAINING_CONFIGS[model_name]
    print(f'\n{"="*70}')
    print(f'TRAINING: {model_name}')
    print(f'{"="*70}')

    cmd = [
        'python', 'scripts/train.py',
        '--model', model_name,
        '--epochs', str(config['epochs']),
        '--batch-size', str(config['batch_size']),
        '--lr', str(config['lr']),
        '--t-max', str(config['t_max']),
        '--normalize',
        '--filter-by-length',
        '--wandb',
        '--wandb-project', WANDB_PROJECT,
        '--wandb-run-name', f'{model_name}_gpu_v1',
        '--wandb-tags', *config['tags'],
    ]

    # Add model-specific hyperparameters
    if config.get('hidden_size'):
        cmd.extend(['--hidden-size', str(config['hidden_size'])])
    if config.get('n_layers'):
        cmd.extend(['--n-layers', str(config['n_layers'])])
    if config.get('dropout') is not None:
        cmd.extend(['--dropout', str(config['dropout'])])

    if WANDB_ENTITY:
        cmd.extend(['--wandb-entity', WANDB_ENTITY])

    print(f'Command: {" ".join(cmd)}')
    print()

    start_time = time.time()

    # Run training with live output
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in process.stdout:
        print(line, end='')

    process.wait()
    elapsed = time.time() - start_time

    results[model_name] = {
        'elapsed_min': elapsed / 60,
        'return_code': process.returncode,
    }

    print(f'\n{model_name} finished in {elapsed/60:.1f} min (exit code {process.returncode})')

# Summary
print(f'\n{"="*70}')
print('TRAINING SUMMARY')
print(f'{"="*70}')
for name, res in results.items():
    status = 'OK' if res['return_code'] == 0 else f'FAILED (code {res["return_code"]})'
    print(f'  {name:20s} — {res["elapsed_min"]:6.1f} min — {status}')

## 6. Copy Results Back to Drive

Copy checkpoints and outputs back to Google Drive so they persist after the Colab session ends.

In [ ]:
import shutil
import os

# Copy checkpoints back to Drive
local_checkpoints = os.path.join(LOCAL_PROJECT, 'outputs', 'checkpoints')
drive_checkpoints = os.path.join(DRIVE_PROJECT_ROOT, 'outputs', 'checkpoints')

if os.path.exists(local_checkpoints):
    os.makedirs(drive_checkpoints, exist_ok=True)
    for f in os.listdir(local_checkpoints):
        src = os.path.join(local_checkpoints, f)
        dst = os.path.join(drive_checkpoints, f)
        print(f'Copying {f} to Drive...')
        shutil.copy2(src, dst)
    print('Checkpoints saved to Drive!')
else:
    print('No checkpoints found.')

# Copy any output figures/results too
local_results = os.path.join(LOCAL_PROJECT, 'outputs', 'results')
drive_results = os.path.join(DRIVE_PROJECT_ROOT, 'outputs', 'results')

if os.path.exists(local_results):
    os.makedirs(drive_results, exist_ok=True)
    for f in os.listdir(local_results):
        shutil.copy2(os.path.join(local_results, f), os.path.join(drive_results, f))
    print('Results saved to Drive!')

## 7. Quick Evaluation (Optional)

Run a quick CER comparison across all trained models.

In [ ]:
import torch
import os

checkpoint_dir = os.path.join(LOCAL_PROJECT, 'outputs', 'checkpoints')

print('Saved checkpoints:')
print('-' * 50)
if os.path.exists(checkpoint_dir):
    for f in sorted(os.listdir(checkpoint_dir)):
        if f.endswith('.pt'):
            path = os.path.join(checkpoint_dir, f)
            ckpt = torch.load(path, map_location='cpu', weights_only=True)
            print(f"  {f:35s}  epoch={ckpt.get('epoch', '?'):>3}  "
                  f"val_cer={ckpt.get('val_cer', '?'):.4f}")
else:
    print('  No checkpoint directory found.')

In [ ]:
# Print W&B dashboard link
entity = WANDB_ENTITY or wandb.api.default_entity
print(f'\nView all runs at:')
print(f'  https://wandb.ai/{entity}/{WANDB_PROJECT}')